##### 실행 환경 설정

In [ ]:
try:
    # Google Drive를 Colab에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # 작업 경로 설정
    WORK_DIR = "/google_drive/Othercomputers/내 Mac/sec09"
    print("\n[작업 폴더 목록]")
    %cd {WORK_DIR}
    !ls
    print()

    # 한글 폰트 설치
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm
    !apt-get -qq install fonts-nanum
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
    # 마이너스 기호 깨짐 방지
    plt.rcParams['axes.unicode_minus'] = False

    # 패키지 설치
    # trl 모듈:
    # - Transformer Reinforcement Learning
    # - Transformer 모델의 파인튜닝을 위한 라이브러리
    !pip install trl
    # torchao 패키지 설치
    # -  PyTorch 모델의 양자화(Quantization)와를 위한 PyTorch 공식 라이브러리
    !pip install --upgrade torchao
    # bitsandbytes 모듈:
    # - 8-bit 및 4-bit 양자화 연산을 지원하는 라이브러리
    !pip install -U "bitsandbytes>=0.46.1"
except Exception:
    # 한글 폰트 설정
    import matplotlib.pyplot as plt
    plt.rcParams['font.family'] = 'Malgun Gothic'
    # 마이너스 기호 깨짐 방지
    plt.rcParams['axes.unicode_minus'] = False

Mounted at /google_drive

[작업 폴더 목록]
/google_drive/Othercomputers/내 Mac/sec09
01_bert_model.ipynb	  03_lora_finetuning_llama.ipynb  runs
02_bert_finetuning.ipynb  datasets

Selecting previously unselected package fonts-nanum.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datase

##### Device 설정

In [ ]:
import gc
import torch

# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

# 큰 모델을 다시 불러오기 전에 이전 객체와 GPU 캐시를 정리하는 함수
def clear_cuda_memory(*variable_names):
    # 참조가 남아 있으면 empty_cache()만 호출해도 메모리가 해제되지 않으므로
    # 먼저 전달받은 전역 변수들을 제거함
    for name in variable_names:
        globals().pop(name, None)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        allocated = torch.cuda.memory_allocated() / 1024 ** 3
        print(f"GPU 메모리 정리 완료 (현재 할당: {allocated:.2f} GB)")

# 같은 런타임에서 '모두 실행'을 다시 눌렀을 때 이전 실행의 큰 객체부터 제거
clear_cuda_memory(
    "trainer", "train_result", "model", "base_model",
    "finetuned_model", "peft_model", "merged_model",
    "inputs", "outputs", "generated", "answer",
)

사용 장치: cuda
GPU 메모리 정리 완료 (현재 할당: 0.00 GB)


##### 허깅 페이스 로그인

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

    from huggingface_hub import login
    login(token=HF_TOKEN)

    print("코랩 HuggingFace 로그인 완료")
except Exception:
    # os 모듈: 운영 체제와 상호 작용하기 위한 라이브러리
    import os
    # huggingface_hub 모듈:
    # - HuggingFace 모델 허브와 상호 작용하기 위한 라이브러리
    # - login 함수: HuggingFace 토큰을 사용하여 로그인하는 함수
    from huggingface_hub import login

    # HuggingFace 토큰으로 로그인
    # - 토큰 발급: https://huggingface.co/settings/tokens
    # - 모델 접근 신청: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
    # - 시스템 환경 변수 추가: HF_TOKEN=발급받은_토큰
    HF_TOKEN = os.environ.get("HF_TOKEN")
    login(token=HF_TOKEN)

    print("로컬 HuggingFace 로그인 완료")

코랩 HuggingFace 로그인 완료


##### 상수 정의

In [ ]:
# os 모듈: 운영 체제와 상호 작용하기 위한 라이브러리
import os

# 파인 튜닝할 모델 이름 설정
BASE_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# 데이터셋 JSONL 파일 경로
DATASET_PATH = f"{WORK_DIR}/datasets/llm_finetuning_train.jsonl"

# 학습 결과가 저장될 디렉토리
SAVED_MODELS_DIR = f"{WORK_DIR}/runs/03/lora_finetuning_llama"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

# SFT 체크포인트가 저장될 디렉토리
# - 학습 중 GPU 크래시/연결 끊김 → 중간부터 재시작 가능용
CHECKPOINTS_DIR = f"{SAVED_MODELS_DIR}/checkpoints"
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

# LoRA 어댑터가 저장될 디렉토리
ADAPTER_DIR = f"{SAVED_MODELS_DIR}/adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

# 기존 모델과 LoRA 어댑터가 병합된 새로운 모델이 저장될 디렉토리
MERGED_MODEL_DIR = f"{SAVED_MODELS_DIR}/merged_model"
os.makedirs(MERGED_MODEL_DIR, exist_ok=True)

##### 모델 로드

In [ ]:
# transformers 모듈:
# - Hugging Face의 트랜스포머 모델과 토크나이저를 불러오기 위한 라이브러리
# - AutoModelForCausalLM: 사전 학습된 인과적 언어 모델을 불러오는 클래스
from transformers import AutoModelForCausalLM

# 기반 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,        # 모델 이름
    dtype=torch.float16,    # 모델을 16비트 부동 소수점으로 로드하여 메모리 사용량을 줄임(기본값: torch.float32)
    low_cpu_mem_usage=True, # 모델 로드 중 CPU 메모리의 중복 사용을 줄임
)

# 모델을 디바이스로 이동 (CPU 또는 GPU)
base_model.to(device)

# K, V 캐시 비활성화
# - 추론 모드 (True):
#     "안녕" 생성 → "하" 생성 → "세" 생성... 처럼 토큰을 하나씩 순차 생성.
#     매번 이전 토큰들의 K,V가 필요하므로 캐싱해서 재사용하므로서 속도 향상
# - 학습 모드 (False):
#     "안녕하세요" 전체 토큰을 한 번에 입력하므로 모든 토큰의 K,V를 동시에 계산.
base_model.config.use_cache = False

# 모델의 전체 파라미터 수 계산
# - base_model.parameters(): 모델의 모든 가중치 텐서에 대한 반복자(iterator)를 반환.
# - numel(): 가중치 텐서의 요소 수를 반환. 예를 들어, 가중치 텐서가 3x3 행렬이면 numel()은 9를 반환
total_params = sum(p.numel() for p in base_model.parameters())
print(f"모델: {BASE_MODEL_NAME}")
print(f"전체 파라미터: {total_params:,}")


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

모델: meta-llama/Llama-3.2-3B-Instruct
전체 파라미터: 3,212,749,824


##### LoRA가 적용된 PEFT 모델 생성

In [ ]:
# peft 모듈: LoRA(저차원 적응) 설정을 위한 라이브러리
# - LoraConfig: LoRA 설정을 정의하는 클래스
# - TaskType: LoRA가 적용될 작업 유형을 정의하는 열거형 클래스
# - get_peft_model: LoRA 설정을 기반으로 PEFT 모델을 생성하는 함수
from peft import LoraConfig, TaskType, get_peft_model

# LoRA 설정
lora_config = LoraConfig(
    # TaskType.CAUSAL_LM: 언어 모델처럼 다음 토큰을 예측하는 작업에 LoRA를 적용
    task_type=TaskType.CAUSAL_LM,
    # 랭크: 랭크 r이 작을수록 학습할 파라미터 수가 줄어들어 효율적이지만, 표현력이 떨어질 수 있음
    r=32,
    # lora_alpha:
    # - LoRA의 스케일링 팩터로, LoRA 업데이트의 크기를 조절하는 하이퍼파라미터.
    # - W' = W + (lora_alpha/r)AB
    # - lora_alpha/r: 64/32=2배로 스케일링(가장 널리 쓰임).
    # - lora_alpha를 정해두고 r이 작을수록 스케일링을 크게, r이 클수록 스케일링이 작아지도록 해서
    # - r이 달라져도 업데이트의 실제 크기를 안정적으로 유지하는게 목적
    lora_alpha=64,
    # lora_dropout:
    # - LoRA 업데이트에 적용되는 드롭아웃 비율
    # - 과적합을 방지하기 위해 LoRA 업데이트에 무작위로 드롭아웃을 적용
    # - 기본값은 0.0(드롭아웃 없음)이며, 일반적으로 0.05(5%)를 가장 많이 사용함
    lora_dropout=0.05,
    # target_modules:
    # - LoRA를 적용할 레이어
    target_modules=[
        "q_proj",       # LoRA를 적용할 Q(쿼리) 레이어 이름
        "v_proj",       # LoRA를 적용할 V(값) 레이어 이름
    ]
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)

# 모델의 학습 가능한 파라미터 수 출력
# - trainable params: LoRA로 추가된 A, B 행렬의 파라미터
# - all params: 기존 모델의 모든 파라미터(동결된 W 포함) + LoRA로 추가된 A, B 행렬의 파라미터
# - trainable%: trainable params가 all params에서 차지하는 비율
model.print_trainable_parameters()


trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


##### 데이터셋 준비

In [ ]:
# JSON 모듈: JSON 데이터를 파싱하고 생성하기 위한 라이브러리
import json

# datasets 모듈:
# - Hugging Face의 다양한 데이터셋을 불러오고 처리하기 위한 라이브러리
# - Dataset: Hugging Face의 데이터셋 객체를 생성하는 클래스
from datasets import Dataset

# JSONL 파일의 각 행을 읽어서 JSON 객체로 파싱하여 리스트에 저장
samples = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    # 라인 단위로 JSON 문자열을 읽음
    for line in f:
        # json.loads(): JSON 문자열을 Python 딕셔너리로 변환하는 함수.
        samples.append(json.loads(line))

# samples를 Dataset 객체로 변환
raw_dataset = Dataset.from_list(samples)

print(f"총 학습 샘플 수: {len(raw_dataset):,}건")

print("\n컬럼목록:", raw_dataset.column_names)

print("\n[첫번째 샘플]")
for msg in raw_dataset[0]["messages"]:
    print(f"{msg['role']:<10}: {msg['content']}")


총 학습 샘플 수: 1,000건

컬럼목록: ['messages']

[첫번째 샘플]
system    : 당신은 친절하고 정중한 자동차 보험 상담사입니다. 질문 의도에 맞게 답변하고, 긴급 상황에서는 바로 필요한 조치를 우선 안내합니다. 모르는 정보는 확인이 필요하다고 안내합니다.
user      : 필요 서류를 알려주세요
assistant : 고객님, 문의주셔서 감사합니다. 필요한 서류는 사고 유형과 청구 항목에 따라 달라질 수 있습니다. 일반적으로는 신청서, 진단서, 영수증 등 관련 증빙 자료가 필요할 수 있습니다. 필요하시면 확인해야 할 항목을 정리해 드리겠습니다.


##### 토크나이저 로드

In [ ]:
# transformers 모듈:
# - Hugging Face의 트랜스포머 모델과 토크나이저를 불러오기 위한 라이브러리
# - AutoTokenizer: 사전 학습된 모델에 맞는 토크나이저를 자동으로 불러오는 클래스
from transformers import AutoTokenizer

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# LLaMA 토크나이저는 기본적으로 pad_token이 없으므로 eos_token을 pad_token으로 설정
# 배치 패딩 처리를 위해 필요
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

##### chat_template 형식으로 변환

In [ ]:
# 샘플을 chat_template 형식으로 변환하는 함수
# chat_template:모델마다 정해진 특수 토큰으로 역할(role)을 구분하는 텍스트 형식
# --------------------------------------------------
# <|begin_of_text|><|start_header_id|>system<|end_header_id|>
#
# 당신은 친절한 자동차 보험 상담사입니다.<|eot_id|><|start_header_id|>user<|end_header_id|>
#
# 사고가 났는데 어떻게 해야 하나요?<|eot_id|><|start_header_id|>assistant<|end_header_id|>
#
# 걱정하지 마세요. 먼저 안전한 곳으로 이동하세요.<|eot_id|>
# --------------------------------------------------
# Llama 3.2의 특수 토큰
# <|begin_of_text|>: 전체 시퀀스의 시작을 나타내는 BOS(Beginning of Sequence) 토큰
# <|start_header_id|>: 역할(role) 헤더 시작
# <|end_header_id|>: 역할(role) 헤더 끝, 이후 두 줄 개행(\n\n) 후 내용이 시작됨
# <|eot_id|>: End of Turn, 해당 역할의 메시지가 끝났음을 나타냄

def format_to_chat(sample):
    text = tokenizer.apply_chat_template(
        # 한 개의 샘플 입력
        sample["messages"],
        # 토크나이저가 텍스트를 토큰으로 변환하는 과정을 수행하지 않도록 설정
        # - 모델에 입력하기 전에 텍스트 형식으로 유지
        # - True, 기본값: 토큰라이징(텍스트를 토큰ID 리스트로 변환)처리해서 바로 모델에 넣을 때 사용
        # - False: 토큰나이징을 하지 않을 때 사용. 학습시 SFTTrainer가 내부적으로 토크나이징을 처리
        tokenize=False,
        # 텍스트 끝에 <|start_header_id|>assistant<|end_header_id|>\n\n를 붙이는지 여부
        # - 학습시(False, 기본값): assistant 완성된 응답을 사용하므로 불필요
        # - 추론시(True): <|start_header_id|>assistant<|end_header_id|>\n\n를 붙여
        #                "지금부터 assistant로서 답하라"고 모델에 신호를 줌
        add_generation_prompt=False,
    )
    # SFTTrainer가 이 "text" 키의 값을 모델 입력으로 사용하기 때문에 "text" 키로 반환
    return {"text": text}

# raw_dataset의 각 샘플을 format_to_chat 함수를 적용하여 chat_template 형식으로 변환
# - remove_columns=raw_dataset.column_names: 원본 컬럼(messages)을 제거하고, "text" 컬럼만 남김
dataset = raw_dataset.map(format_to_chat, remove_columns=raw_dataset.column_names)

print("[chat_template 형식으로 변환된 샘플]")
print(dataset[0]["text"])


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

[chat_template 형식으로 변환된 샘플]
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 24 Jul 2026

당신은 친절하고 정중한 자동차 보험 상담사입니다. 질문 의도에 맞게 답변하고, 긴급 상황에서는 바로 필요한 조치를 우선 안내합니다. 모르는 정보는 확인이 필요하다고 안내합니다.<|eot_id|><|start_header_id|>user<|end_header_id|>

필요 서류를 알려주세요<|eot_id|><|start_header_id|>assistant<|end_header_id|>

고객님, 문의주셔서 감사합니다. 필요한 서류는 사고 유형과 청구 항목에 따라 달라질 수 있습니다. 일반적으로는 신청서, 진단서, 영수증 등 관련 증빙 자료가 필요할 수 있습니다. 필요하시면 확인해야 할 항목을 정리해 드리겠습니다.<|eot_id|>


##### SFT 학습 설정 (SFTConfig)

In [ ]:
# trl 모듈: 트랜스포머 모델을 미세 조정하기 위한 라이브러리
# - SFTConfig: SFT 훈련 설정을 정의하는 클래스
from trl import SFTConfig

# SFT 학습 설정
sft_config = SFTConfig(
    # 학습 결과 저장 디렉토리
    output_dir=CHECKPOINTS_DIR,
    # 전체 데이터셋을 반복 학습 횟수
    # 기본값 3, 대용량 데이터:1~3, 중간 데이터(1천~1만):3~5, 소량 데이터(1천 이하):5~10
    num_train_epochs=5,
    # 배치 관련 크기
    # - per_device_train_batch_size = 4: GPU에 한 번에 올리는 실제 배치 크기
    # - gradient_accumulation_steps = 4: 기울기를 누적할 횟수
    # - 1배치(샘플 4) 기울기를 계산하고 이를 4번 누적해서 한번의 파라미터 업데이트
    per_device_train_batch_size=4,  # GPU 메모리 제한으로 인해 작은 배치 사이즈 사용
    gradient_accumulation_steps=4,  # 4개의 배치를 누적하여 기울기를 계산
    # 학습률
    # - LoRA Fine-tuning: 1e-4 ~ 5e-4
    learning_rate=3e-4,
    # 학습률 스케줄러 유형
    # - linear: 학습률이 선형적으로 감소. 기본값
    # - cosine: 학습률이 코사인 곡선으로 부드럽게 감소. LLM 파인 튜닝에서 가장 많이 사용됨
    lr_scheduler_type="cosine",
    # 학습 초기에 학습률을 0에서 목표값(learning_rate)까지 서서히 올리는 비율
    # - 전체 학습 스텝의 3%를 워밍업 단계로 사용하여 학습 초기에 안정적으로 수렴하도록 도움
    warmup_ratio=0.03,
    # 모델 입력 시퀀스의 최대 길이
    max_length=512,
    # 로그 출력 빈도
    logging_steps=10,
    # 체크포인트 저장 전략
    # - 체크포인트: 학습 중간에 모델의 가중치와 옵티마이저 상태를 저장해둔 스냅샷
    # - epoch: 각 에폭이 끝날 때마다 체크포인트 저장
    save_strategy="epoch",
    # FP16 사용 여부
    # - false: 기본값, FP32. 메모리 사용량이 많지만 안정적인 학습을 제공
    # - true: 메모리 사용량이 절반으로 줄어들어 더 큰 배치 사이즈나 긴 시퀀스를 사용할 수 있어 권장됨.
    #         LLM 파인 튜닝에서는 GPU에서 FP16을 사실상 표준
    #         CPU는 FP32만 지원하므로 GPU에서 학습할 때만 True로 설정하는 것이 일반적
    fp16=torch.cuda.is_available(),
    # 옵티마이저 설정
    # - LLM 파인 튜닝에서 가장 널리 사용되는 옵티마이저는 AdamW
    # - OptimizerNames enum에 정의되어 있는 값을 사용해야 함 adamw(x), adamw_tourch(o)
    optim="adamw_torch",
    # FTTrainer의 내부 전처리를 건너뛰어 collator가 text 키를 가진 데이터셋을 직접 처리하도록 설정
    dataset_kwargs={"skip_prepare_dataset": True},
    # SFTTrainer가 데이터셋의 사용되지 않는 컬럼을 제거하지 않도록 설정
    remove_unused_columns=False
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


##### 모델 입력 배치 생성(collator)

In [ ]:
# ResponseOnlyCollator 클래스 정의
# - SFTTrainer가 모델에 입력할 배치를 생성하는 역할
# - assistant 응답 토큰에만 Loss를 계산하도록 레이블을 마스킹도 함
# - response_template 이전 토큰(system + user)은 labels=-100으로 마스킹하여 Loss 계산에서 제외
class ResponseOnlyCollator:
    # response_template: assistant 응답 위치를 알려주는 토큰 시퀀스
    # - LLaMA 3.2: "<|start_header_id|>assistant<|end_header_id|>\n\n"
    # tokenizer: response_template을 토큰 ID 시퀀스로 변환하기 위해 필요
    def __init__(self, response_template: str, tokenizer, max_length: int):
        # response_template를 토큰 ID 리스트로 변환
        # - "<|start_header_id|>assistant<|end_header_id|>\n\n" → [128006, 78191, 128007, 271]
        # - add_special_tokens=False: [CLS], [SEP] 등의 특수 토큰을 추가하지 않음
        self.response_ids = tokenizer.encode(response_template, add_special_tokens=False)
        self.tokenizer = tokenizer
        self.max_length = max_length

    # SFTTrainer가 넘겨주는 1배치을 받아서, 패딩 및 마스킹을 적용한 새로운 1배치를 생성
    def __call__(self, samples):
        # samples는 [{'text': '...'}] 형태의 리스트

        # 1. 각 샘플의 'text'를 토큰화
        tokenized_batch_items = []
        for sample in samples:
            tokenized = self.tokenizer(
                sample["text"],
                truncation=True,
                max_length=self.max_length,
                return_tensors=None, # 패딩을 나중에 일괄 적용하기 위해 리스트로 반환
                padding=False # 개별 샘플에 패딩 적용 안 함
            )
            tokenized_batch_items.append(tokenized)

        # 2. 토큰화된 배치에 패딩 적용
        batch = self.tokenizer.pad(
            tokenized_batch_items,
            padding="longest",  # 배치 내 가장 긴 시퀀스에 맞춰 패딩
            return_tensors="pt",
        )

        # 정답 레이블 텐서 생성
        # - 모델이 예측한 토큰과 비교하여 Loss 계산에 사용
        # - input_ids를 복사해서 생성
        labels = batch["input_ids"].clone()
        # 배치 내 각 샘플에 대해 반복
        for i in range(len(labels)):
            # i번째 샘플의 토큰 ID 시퀀스를 리스트로 변환
            seq = labels[i].tolist()
            # response_template 시작 위치 찾기
            start = None
            for j in range(len(seq) - len(self.response_ids) + 1):
                # j 위치에서 response_template_ids과 일치하는지 확인
                if seq[j : j + len(self.response_ids)] == self.response_ids:
                    # response_template_ids 바로 뒤의 위치 저장
                    start = j + len(self.response_ids)
                    break
            if start is None:
                # response_template을 찾지 못한 경우(assistant 응답이 없는 잘못된 샘플)
                # 전체 labels를 -100으로 채워 해당 샘플은 학습에 기여하지 않도록 처리
                # -100: PyTorch CrossEntropyLoss가 Loss 계산을 무시하는 특수 값
                labels[i] = -100
            else:
                # response_template 포함해서 이전 토큰ID들을 모두 -100으로 마스킹
                # system + user + "<|start_header_id|>assistant<|end_header_id|>\n\n" 부분은
                # -100으로 마스킹하여 Loss 계산에서 제외
                # assistant 응답 내용부터 Loss가 계산되도록 함
                labels[i, :start] = -100

        # 정답 레이블 텐서를 batch 딕셔너리에 "labels" 키로 추가
        batch["labels"] = labels

        # 배치 반환
        # batch = {
        #   "input_ids":      [[토큰ID, 토큰ID, ..., 토큰ID,  PAD,   PAD],   ...],
        #   "attention_mask": [[1,      1,     ..., 1,        0,     0],  ...],
        #   "labels":  [[-100,  -100,  ..., 토큰ID, ..., 토큰ID], ...],
        # }
        return batch

# 모델 입력 배치 생성을 위한 데이터 콜레이터 생성
collator = ResponseOnlyCollator(
    # assistant 응답이 시작되는 부분을 나타내는 템플릿
    response_template="<|start_header_id|>assistant<|end_header_id|>\n\n",
    # 토크나이저
    tokenizer=tokenizer,
    # SFTConfig에 정의된 max_length 사용
    max_length=sft_config.max_length,
)

##### SFT 학습기 생성 (SFTTrainer)

In [ ]:
# trl 모듈: 트랜스포머 모델을 미세 조정하기 위한 라이브러리
# - SFT(Supervised Fine-Tuning, 지도학습 파인튜닝) 방식으로 훈련하기 위한 클래스
# - SFTConfig: SFT 훈련 설정을 정의하는 클래스
from trl import SFTTrainer

# SFTTrainer 생성
# - 트랜스포머 모델을 SFT(Supervised Fine-Tuning, 지도 학습 방식)으로 훈련하기 위한 객체
trainer = SFTTrainer(
    # 학습할 모델
    model=model,
    # 학습 데이터셋 설정
    train_dataset=dataset,
    # 토크나이저 설정
    processing_class=tokenizer,
    # 학습 설정
    args=sft_config,
    # 모델 입력 배치 생성을 위한 데이터 콜레이터
    data_collator=collator
)
print("SFTTrainer 준비 완료")

SFTTrainer 준비 완료


##### 학습하기

In [ ]:
# SFTTrainer 내부 학습 과정
# - Dataset (text 문자열들)
#     ↓ SFTTrainer 내부에서 토크나이징 (text → input_ids 리스트)
#     ↓
# - DataLoader.get_train_dataloader()
#     ↓ Dataset에서 batch_size(=4)개 샘플을 꺼냄
#     ↓ collate_fn=collator 로 모델 입력을 위한 배치 조립 요청
#     ↓
# - collator.__call__(samples)  ← 여기서 패딩·레이블 마스킹 적용
#     ↓
# - 모델에 배치 전달 → Loss 계산 → 역전파
train_result = trainer.train()

# 최상위 필드
#----------------------------------------
# 필드          | 타입     | 설명
#----------------------------------------
# global_step	  int	    완료된 총 학습 스텝 수
# training_loss	  float	    전체 학습 평균 손실값
# metrics	      dict	    상세 학습 지표 딕셔너리

# metrics 딕셔너리 주요 키
#----------------------------------------
# 키                       | 설명
#----------------------------------------
# train_runtime	            총 학습 시간 (초)
# train_samples_per_second	초당 처리 샘플 수
# train_steps_per_second	초당 처리 스텝 수
# train_loss	            평균 손실값 (training_loss와 동일)
# epoch	                    완료된 에폭 수
# total_flos	            총 부동소수점 연산 수 (FLOPs)

print(f"총 학습 시간: {train_result.metrics['train_runtime']:.1f}초")
print(f"최종 훈련 손실: {train_result.training_loss:.4f}")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,3.732815
20,1.713799
30,1.107494
40,0.743406
50,0.474155
60,0.303677
70,0.244066
80,0.191936
90,0.147245
100,0.113121


총 학습 시간: 1547.7초
최종 훈련 손실: 0.3145


##### LoRA 어댑터 가중치(A, B 행렬) 저장

In [ ]:
# LoRA 어댑터만 저장 (기반 모델 제외)
model.save_pretrained(ADAPTER_DIR)

##### 답변 생성 함수 정의

In [ ]:
# 사용자 쿼리에 대해 모델이 응답을 생성하는 함수
def generate_answer(model, tokenizer, user_query, max_new_tokens=128):
    # 시스템 프롬프트 정의: 자동차 보험 상담사 역할을 모델에 명확히 지시하기 위한 텍스트
    SYSTEM_PROMPT = (
        "당신은 친절하고 전문적인 자동차 보험 상담사입니다. "
        "고객의 질문에 공감하며 따뜻하고 신뢰감 있는 말투로 응대합니다. "
        "반드시 한국어로 답변해주세요."
    )

    # 모델에 입력될 메시지 리스트 생성
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    # 메시지 리스트를 chat_template 형식의 텍스트로 변환
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        # 추론 시 모델이 assistant 역할로 응답을 생성하도록 템플릿 끝에
        # <|start_header_id|>assistant<|end_header_id|>\n\n를 추가
        add_generation_prompt=True,
    )

    # chat_template 형식으로 변환된 텍스트를 모델 입력으로 사용할 수 있도록
    # 토크나이징하여 PyTorch 텐서로 변환
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # 추론해서 모델이 다음 토큰을 하나씩 순차적으로 생성
    with torch.no_grad():
        # model.generate(): 모델이 입력 시퀀스에 이어서 새로운 토큰을 생성하도록 하는 함수
        outputs = model.generate(
            # 모델 입력 텐서
            # inputs = {
            #   "input_ids":      tensor([[128000, 12345, ..., 54321]]),
            #   "attention_mask": tensor([[1, 1, 1, ...1]]), #단일 입력이라 전부 1
            # }
            **inputs,
            # 모델이 생성할 최대 토큰 수 제한
            max_new_tokens=max_new_tokens,
            # 샘플링 여부 설정
            # - False(기본값):
            #     매 스텝마다 가장 높은 확률의 토큰을 선택하여 생성.
            #     동일한 입력 → 동일한 출력(결정적), 정확성과 일관성이 중요할 때 권장됨
            #     temperature, top_p는 무시됨
            # - True:
            #     확률에 따라 토큰을 무작위로 선택하여 생성. 다양성이 증가
            #     temperature, top_p는 적용됨
            do_sample=False,
            # temperature: 샘플링 시 생성의 무작위성 정도를 조절하는 하이퍼파라미터
            # temperature=0.7,
            # top_p: 다음 토큰을 선택할 때 고려할 누적 확률 임계값
            # - 0.9: 확률이 높은 상위 토큰들의 누적 확률이 90%가 될 때까지 토큰을 고려하여 샘플링
            # top_p=0.9,
            # 같은 토큰이 반복되는 것을 방지하기 위한 패널티
            # - 1.0	패널티 없음 (기본값)
            # - 1.1	약한 반복 억제 (자연스러운 수준)
            # - 1.3~1.5	강한 반복 억제
            # - 2.0+ 과도한 억제. 문장이 어색해질 수 있음
            repetition_penalty=1.1,
            # 패딩 토큰 ID 설정 (모델 어휘에서 PAD 토큰이 없는 경우 EOS 토큰 ID 사용)
            # - 단일 입력일 경우 패딩이 필요 없지만, 모델이 패딩 토큰 ID를 요구하는 경우를 대비하여 설정
            pad_token_id=tokenizer.eos_token_id,
        )

    # 모델이 생성한 토큰 시퀀스에서 입력 토큰을 제외한 새로 생성된 토큰 부분만 추출하고 CPU로 이동
    # - inputs["input_ids"].shape[1]: 입력 토큰 시퀀스의 길이
    generated = outputs[0][inputs["input_ids"].shape[1]:].cpu()

    # tokenizer.decode()를 사용하여 생성된 토큰 시퀀스를 사람이 읽을 수 있는 텍스트로 변환
    return tokenizer.decode(generated, skip_special_tokens=True)

##### 평가하기

In [ ]:
# transformers 모듈: Hugging Face의 트랜스포머 모델과 토크나이저를 불러오기 위한 라이브러리
# - AutoTokenizer: 사전 학습된 모델에 맞는 토크나이저를 자동으로 불러오는 클래스
# - AutoModelForCausalLM: 사전 학습된 언어 모델을 불러오는 클래스, 특히 인과 언어 모델링 작업에 사용
from transformers import AutoTokenizer, AutoModelForCausalLM

# peft 모듈: LoRA(저차원 적응) 설정을 위한 라이브러리
# - PeftModel: LoRA로 미세 조정된 모델을 나타내는 클래스
from peft import PeftModel

# 학습에 사용한 Trainer와 모델을 먼저 해제하여 T4 GPU 메모리를 확보
clear_cuda_memory("trainer", "train_result", "model", "base_model")

# 기반 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# LoRA 어댑터와 병합된 모델 얻기
finetuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# LLaMA 토크나이저는 기본적으로 pad_token이 없으므로 eos_token을 pad_token으로 설정
# 배치 패딩 처리를 위해 필요
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델을 평가 모드로 설정하여 추론 준비
finetuned_model.eval()

# 추론 모드에서는 K, V 캐싱하여 속도 향상. 대신 메모리 사용량 증가됨
finetuned_model.config.use_cache = True

# 자동차 보험 상담 테스트 쿼리
user_queries = [
    "사고가 났는데 어떻게 해야 하나요?",
    "보험료를 낮출 방법이 있을까요?",
    "배터리가 방전됐어요.",
]

# 어뎁터 탈착(기반 모델)과 어뎁터 부착(파인튜닝된 모델)의 응답 비교
for query in user_queries:
    print(f"[질문] {query}")
    print()

    # Before: LoRA 어댑터 비활성화 → 기반 모델 동작
    with finetuned_model.disable_adapter():
        base_response = generate_answer(finetuned_model, tokenizer, query)
    print(f"[어댑터 탈착] 기반 모델:\n  {base_response}")
    print()

    # After: 저장된 파인튜닝 모델
    finetuned_response = generate_answer(finetuned_model, tokenizer, query)
    print(f"[어댑터 부착] 파인튜닝 모델:\n  {finetuned_response}")
    print("-" * 60)


GPU 메모리 정리 완료 (현재 할당: 0.02 GB)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[질문] 사고가 났는데 어떻게 해야 하나요?



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[어댑터 탈착] 기반 모델:
  안녕하세요, tôi rất tiết rằng bạn đang gặp khó khăn. 사고가 발생했을 때는 우선적으로 안전을 위하여 차량을 멈추고, 모든 사람을 확인하세요. 그리고 police나 의료 전문가와 연락을 취하여 상황을 파악하는 것이 중요합니다.

그리고, 이 경우에는 즉시 보험 회사에 연락을 취하여 사고를รายงาน하고, 필요한 경우 보험 coverage를 확인하는 것이 좋습니다. 또한, 경찰과 의사에게 의견을 얻으며, 필요한 경우 치료를 받으시는 것을 추천드립니다.

또한, 사고

[어댑터 부착] 파인튜닝 모델:
  고객님, 문의주셔서 감사합니다. 사고 직후에는 현장 안전 확보와 부상 여부 확인이 가장 중요합니다. 그다음 사고 경위와 상대 차량 정보를 정리한 뒤 보험사에 접수하시면 절차 진행에 도움이 됩니다. 필요 시 사고 사진과 상대 차량 정보를 함께 정리해 두시면 접수에 도움이 됩니다.
------------------------------------------------------------
[질문] 보험료를 낮출 방법이 있을까요?

[어댑터 탈착] 기반 모델:
  안녕하세요! 저희는 आपक을 도와드릴게요.

보험이란 예상치 못한 사고나 재화/재산의 손실 등에 대한 보상을 제공하는 계약입니다. 보험료는 보험가액과 보험료율에 따라 결정되며, 보험료율은 보험사마다 다르며, 보험사마다 다른 요인들이 influencing factor로 작용할 수 있습니다.

보험료를 낮추기 위해서는 다음과 같은 방법이 있습니다:

1. **보험가액을 높여서 보험료율을 낮추

[어댑터 부착] 파인튜닝 모델:
  고객님, 문의주셔서 감사합니다. 자동차 보험료는 가입 조건과 특약 구성에 따라 달라지므로 일부 항목을 조정하면 부담을 낮출 수 있습니다. 다만 보장이 줄어들 수 있으니 변경 전후 내용을 함께 비교해 보시는 것을 권해드립니다. 추가 정보가 있으면 더 구체적으로 안내드릴 수 있습니다.
-------------------------

##### 파인튜닝된 모델 전체 저장

In [17]:
# torch 모듈: PyTorch 라이브러리로, 텐서 연산과 딥 러닝 모델 구축에 사용
import torch

# transformers 모듈: Hugging Face의 트랜스포머 모델과 토크나이저를 불러오기 위한 라이브러리
from transformers import AutoTokenizer, AutoModelForCausalLM

# peft 모듈: LoRA(저차원 적응) 설정을 위한 라이브러리
from peft import PeftModel

# 비교 추론에 사용한 모델을 해제한 뒤 병합용 모델을 로드
clear_cuda_memory("finetuned_model", "base_model", "base_response", "finetuned_response")

# 기반 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# LoRA 어댑터와 병합된 모델 로드
peft_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# LoRA 어댑터를 기반 모델에 병합하여 새로운 모델 생성
# - merge: LoRA 어댑터의 가중치를 기반 모델에 통합하여 새로운 모델 생성
# - unload: LoRA 어댑터를 메모리에서 제거하여 메모리 사용량 최적화
merged_model = peft_model.merge_and_unload(safe_merge=True)

# 새로운 모델 저장
merged_model.save_pretrained(MERGED_MODEL_DIR)
print(f"모델 저장 완료 ({MERGED_MODEL_DIR})")

# 토크나이저도 함께 저장
# - 저장된 모델과 동일한 디렉토리에 토크나이저도 저장하여
# - 모델과 토크나이저가 일관된 상태로 유지되도록 함
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained(MERGED_MODEL_DIR)
print(f"토크나이저 저장 완료 ({MERGED_MODEL_DIR})")

GPU 메모리 정리 완료 (현재 할당: 0.02 GB)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료 (/google_drive/Othercomputers/내 Mac/sec09/runs/03/lora_finetuning_llama/merged_model)
토크나이저 저장 완료 (/google_drive/Othercomputers/내 Mac/sec09/runs/03/lora_finetuning_llama/merged_model)


##### 추론하기

In [18]:
# torch 모듈: PyTorch 라이브러리로, 텐서 연산과 딥 러닝 모델 구축에 사용
import torch

# transformers 모듈: Hugging Face의 트랜스포머 모델과 토크나이저를 불러오기 위한 라이브러리
from transformers import AutoTokenizer, AutoModelForCausalLM

# 병합 작업에 사용한 모델들을 해제한 뒤 저장된 병합 모델을 다시 로드
clear_cuda_memory("merged_model", "peft_model", "base_model")

# 모델 로드
merged_model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)

# 모델을 평가 모드로 설정하여 추론 준비
merged_model.eval()
merged_model.config.use_cache = True

# 시스템 프롬프트
SYSTEM_PROMPT = (
    "당신은 친절하고 전문적인 자동차 보험 상담사입니다. "
    "고객의 질문에 공감하며 따뜻하고 신뢰감 있는 말투로 응대합니다. "
    "반드시 한국어로 답변해주세요."
)

# 사용자 질문
user_query = "사고가 났는데 어떻게 해야 하나요?"

# 메시지 구성
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_query},
]

# 메시지를 chat_template 형식으로 변환
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# 토크나이징
inputs = tokenizer(input_text, return_tensors="pt").to(merged_model.device)

# 질문하기
print(f"[질문] {user_query}")
with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,    # 가장 높은 확률의 토큰을 선택
        repetition_penalty=1.1, # 같은 토큰이 반복되는 것을 방지하기 위한 패널티
        pad_token_id=tokenizer.eos_token_id, # 패딩 토큰 ID 설정
    )

# 답변받기
generated = outputs[0][inputs["input_ids"].shape[1]:].cpu()
answer = tokenizer.decode(generated, skip_special_tokens=True)
print(f"[답변] {answer}")

GPU 메모리 정리 완료 (현재 할당: 0.02 GB)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[질문] 사고가 났는데 어떻게 해야 하나요?
[답변] 고객님, 문의주셔서 감사합니다. 사고 직후에는 현장 안전 확보와 부상 여부 확인이 가장 중요합니다. 그다음 사고 경위와 상대 차량 정보를 정리한 뒤 보험사에 접수하시면 절차 진행에 도움이 됩니다. 필요 시 사고 사진과 상대 차량 정보를 함께 정리해 두시면 접수에 도움이 됩니다.


##### 양자화 적용 모델 사용하기 (GPU 메모리 절약)

In [19]:
# torch 모듈: PyTorch 라이브러리로, 텐서 연산과 딥 러닝 모델 구축에 사용
import torch

# transformers 모듈: Hugging Face의 트랜스포머 모델과 토크나이저를 불러오기 위한 라이브러리
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

# 모듈 경고 메시지 출력 안함
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")

# FP16 추론 모델과 생성 텐서를 해제한 뒤 4-bit 모델을 로드
clear_cuda_memory("merged_model", "inputs", "outputs", "generated", "answer")

# 4-bit 양자화 추론 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 4-bit 양자화 활성화
    bnb_4bit_quant_type="nf4",              # NormalFloat 4-bit 사용 (LLM에 최적화)
    bnb_4bit_compute_dtype=torch.float16,   # 연산은 FP16으로 수행
    bnb_4bit_use_double_quant=True,         # 이중 양자화로 메모리 추가 절약
)

# 모델 로드 (로드 시 양자화 적용)
merged_model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,                # 병합된 모델이 저장된 디렉토리
    quantization_config=bnb_config,  # 로드 시 양자화 설정 적용
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)

# 모델을 평가 모드로 설정하여 추론 준비
merged_model.eval()
merged_model.config.use_cache = True

# 시스템 프롬프트
SYSTEM_PROMPT = (
    "당신은 친절하고 전문적인 자동차 보험 상담사입니다. "
    "고객의 질문에 공감하며 따뜻하고 신뢰감 있는 말투로 응대합니다. "
    "반드시 한국어로 답변해주세요."
)

# 사용자 질문
user_query = "사고가 났는데 어떻게 해야 하나요?"

# 메시지 구성
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_query},
]

# 메시지를 chat_template 형식으로 변환
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# 토크나이징
inputs = tokenizer(input_text, return_tensors="pt").to(merged_model.device)

# 질문하기
print(f"[질문] {user_query}")
with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

# 답변받기
generated = outputs[0][inputs["input_ids"].shape[1]:].cpu()
answer = tokenizer.decode(generated, skip_special_tokens=True)
print(f"[답변] {answer}")

GPU 메모리 정리 완료 (현재 할당: 0.02 GB)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[질문] 사고가 났는데 어떻게 해야 하나요?
[답변] 고객님, 문의주셔서 감사합니다. 사고 직후에는 현장 안전을 확보하고 부상 여부를 확인하시는 것이 가장 중요합니다. 이후 사고 경위와 상대 차량 정보를 정리한 뒤 보험사에 접수하시면 보다 원활하게 진행하실 수 있습니다. 필요하시면 구체적으로 설명해 드리겠습니다.
